In [0]:
# Databricks Delta Live Tables (DLT) - Dashboard Semantic Views
# Catalog: maven_market_uc
# Schema:  gold
#
# NOTE: This notebook adds DLT views only. No column names or parameters are changed.

import dlt
from pyspark.sql import functions as F

CATALOG = "maven_market_uc"
GOLD = f"{CATALOG}.gold"

@dlt.view(name="v_exec_kpi_cards")
def v_exec_kpi_cards():
    return spark.read.table(f"{GOLD}.kpi_executive_summary").select(
        "year",
        "total_revenue",
        "total_profit",
        "profit_margin_pct",
        "active_customers",
        "total_transactions",
        "avg_transaction_value",
        "return_rate"
    )

@dlt.view(name="v_exec_monthly_trend")
def v_exec_monthly_trend():
    return (
        spark.read.table(f"{GOLD}.agg_monthly_sales")
        .groupBy("year","month","month_name","quarter","year_month")
        .agg(
            F.sum("total_revenue").alias("total_revenue"),
            F.sum("total_profit").alias("total_profit"),
            (F.sum("total_profit")/F.nullif(F.sum("total_revenue"),F.lit(0))*100).alias("profit_margin_pct")
        )
    )

@dlt.view(name="v_exec_revenue_by_region_top10")
def v_exec_revenue_by_region_top10():
    return (
        spark.read.table(f"{GOLD}.agg_store_performance")
        .groupBy("sales_region")
        .agg(
            F.sum("total_revenue").alias("total_revenue"),
            F.sum("total_profit").alias("total_profit")
        )
        .orderBy(F.desc("total_revenue"))
        .limit(10)
    )

@dlt.view(name="v_exec_top_products_top10")
def v_exec_top_products_top10():
    return (
        spark.read.table(f"{GOLD}.agg_product_performance")
        .select(
            "product_id",
            "product_brand",
            "product_name",
            "gross_revenue",
            "gross_profit",
            "units_sold",
            "units_returned",
            "return_rate"
        )
        .orderBy(F.desc("gross_revenue"))
        .limit(10)
    )

@dlt.view(name="v_regional_revenue_by_store")
def v_regional_revenue_by_store():
    return spark.read.table(f"{GOLD}.agg_store_performance").select(
        "store_id",
        "store_name",
        "sales_region",
        "total_revenue",
        "total_profit",
        "return_rate",
        "avg_transaction_value",
        "unique_customers",
        "total_transactions"
    )

@dlt.view(name="v_regional_monthly_trend")
def v_regional_monthly_trend():
    return (
        spark.read.table(f"{GOLD}.agg_monthly_sales")
        .groupBy("year","month","month_name","quarter","year_month","store_id")
        .agg(
            F.sum("total_revenue").alias("total_revenue"),
            F.sum("total_profit").alias("total_profit")
        )
    )

@dlt.view(name="v_dq_daily_volume_freshness")
def v_dq_daily_volume_freshness():
    return (
        spark.read.table(f"{GOLD}.agg_daily_sales")
        .groupBy("transaction_date")
        .agg(
            F.countDistinct("store_id").alias("stores_reporting"),
            F.sum("total_transactions").alias("total_transactions"),
            F.sum("total_units").alias("total_units"),
            F.sum("total_revenue").alias("total_revenue")
        )
        .orderBy(F.desc("transaction_date"))
    )